In [1]:
from datasets.cross_tissue_atlas import CrossTissueDataset
import os

import hydra
from omegaconf import OmegaConf

import torch
import numpy as np
from geomloss import SamplesLoss

# silence all warnings
import warnings
warnings.filterwarnings("ignore")

import seaborn as sns
import matplotlib.pyplot as plt

/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/home/002/gokulg/miniforge3/envs/gde/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [ ]:
n_pcs = 2

train_set = CrossTissueDataset(
    root="data",
    split="train",
    n_pcs=n_pcs
)

test_set = CrossTissueDataset(
    root="data",
    split="test",
    n_pcs=n_pcs
)

In [ ]:
adata = test_set.adata
train_donors = train_set.donors
test_donors = test_set.donors
print(train_donors)
print(test_donors)

['GTEX-12BJ1', 'GTEX-13N11', 'GTEX-144GM', 'GTEX-145ME', 'GTEX-15CHR', 'GTEX-15EOM', 'GTEX-15RIE', 'GTEX-15SB6']
['GTEX-16BQI', 'GTEX-1CAMR', 'GTEX-1CAMS', 'GTEX-1HSMQ', 'GTEX-1I1GU', 'GTEX-1ICG6', 'GTEX-1MCC2', 'GTEX-1R9PN']


In [ ]:
donor_centroids = {}
for donor in test_donors:
    donor_data = test_set.adata[test_set.adata.obs['donor_id'] == donor]
    centroid = np.nan_to_num(donor_data.obsm['X_pca']).mean(axis=0).flatten()
    donor_centroids[donor] = centroid
for donor in train_donors:
    donor_data = train_set.adata[train_set.adata.obs['donor_id'] == donor]
    centroid = np.nan_to_num(donor_data.obsm['X_pca']).mean(axis=0).flatten()
    donor_centroids[donor] = centroid

# for each test donor, get the nearest train donor
test_to_train_donor = {}
for test_donor in test_donors:
    test_centroid = donor_centroids[test_donor]
    nearest_train_donor = None
    nearest_distance = float('inf')
    for train_donor in train_donors:
        train_centroid = donor_centroids[train_donor]
        distance = torch.norm(torch.tensor(test_centroid) - torch.tensor(train_centroid)).item()
        if distance < nearest_distance:
            nearest_distance = distance
            nearest_train_donor = train_donor
    test_to_train_donor[test_donor] = nearest_train_donor

print("Test to Train Donor Mapping:")
for test_donor, train_donor in test_to_train_donor.items():
    print(f"{test_donor} -> {train_donor}")

Test to Train Donor Mapping:
GTEX-16BQI -> GTEX-15SB6
GTEX-1CAMR -> GTEX-144GM
GTEX-1CAMS -> GTEX-144GM
GTEX-1HSMQ -> GTEX-15RIE
GTEX-1I1GU -> GTEX-15SB6
GTEX-1ICG6 -> GTEX-144GM
GTEX-1MCC2 -> GTEX-144GM
GTEX-1R9PN -> GTEX-144GM


In [ ]:
output_dir = '/orcd/data/omarabu/001/gokul/CoupledDistributionEmbeddings/outputs/'

configs = os.listdir(output_dir)

config_name = [x for x in configs if x.startswith('crosstissue_gnn_energy_fm_')][0]
print(config_name)
config_path = os.path.join(output_dir, config_name, 'config.yaml')
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config not found at {config_path}")

config = OmegaConf.load(config_path)

best_model_path = os.path.join(output_dir, config_name, 'best_model.pt')
if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Best model not found at {best_model_path}")

dct_encoder = hydra.utils.instantiate(config.encoder)
dct_generator = hydra.utils.instantiate(config.generator)

device = 'cuda'
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

dct_encoder.load_state_dict(checkpoint['encoder_state_dict'])
dct_generator.load_state_dict(checkpoint['generator_state_dict'])

epoch = checkpoint.get('epoch', 'unknown')
loss = checkpoint.get('loss', float('nan'))

print(epoch)

dct_encoder.to(device)
dct_generator.to(device)
dct_encoder.eval()
dct_generator.eval();

crosstissue_gnn_energy_fm_d767f47de937c79a909fe7bb8fc646fe
510


In [ ]:
energy = SamplesLoss("energy")

energy_dists = []

for p in range(8):

    for _ in range(100):

        batch = test_set[p]

        source_samples = batch['source_samples'].to(device).unsqueeze(0)
        target_samples = batch['target_samples'].to(device).unsqueeze(0)

        with torch.no_grad():
            source_latent = dct_encoder(source_samples)
            target_latent = dct_encoder(target_samples)

            # print(source_latent.shape)

            samples = dct_generator.sample(source_samples.reshape(-1, n_pcs), 
                                           source_latent, target_latent,
                                           num_steps=5)

        e_dist = energy(samples.squeeze(0), target_samples.squeeze(0)).item()
        energy_dists.append(e_dist)

print("mean Energy Distance:", np.mean(energy_dists))
print('s.e.m', np.std(energy_dists) / np.sqrt(len(energy_dists)))

mean Energy Distance: 0.038245342820882794
s.e.m 0.0009289238552384489


In [ ]:
import random

# Select random source and target donors from test set
# source_donor_idx = random.randint(0, len(test_donors) - 1)
# target_donor_idx = random.randint(0, len(test_donors) - 1)

source_donor = 'GTEX-1HSMQ'#test_donors[source_donor_idx]
target_donor = 'GTEX-1MCC2'#test_donors[target_donor_idx]

adata_indices_src = test_set.donor_indices['GTEX-1HSMQ']
adata_indices_tgt = test_set.donor_indices['GTEX-1MCC2']

print(f"Source donor: {source_donor}")
print(f"Target donor: {target_donor}")

# Get ALL cells from source and target donors
# source_donor_data = test_set.adata[test_set.adata.obs['donor_id'] == source_donor]
# target_donor_data = test_set.adata[test_set.adata.obs['donor_id'] == target_donor]
source_donor_data = test_set.adata[adata_indices_src]
target_donor_data = test_set.adata[adata_indices_tgt]

# subsample to 5k source cells if there are more than 5k
if source_donor_data.shape[0] > 1000:
    source_donor_data = source_donor_data[np.random.choice(source_donor_data.shape[0], 1000, replace=True), :]

source_samples = torch.tensor(np.nan_to_num(source_donor_data.obsm['X_pca']), dtype=torch.float32).to(device).unsqueeze(0)
target_samples = torch.tensor(np.nan_to_num(target_donor_data.obsm['X_pca']), dtype=torch.float32).to(device).unsqueeze(0)

print(f"Source samples shape: {source_samples.shape}")
print(f"Target samples shape: {target_samples.shape}")

# Get true target distribution
true_target_pcs = np.nan_to_num(target_donor_data.obsm['X_pca'])

# Generate trajectories
with torch.no_grad():
    source_latent = dct_encoder(source_samples)
    target_latent = dct_encoder(target_samples)
    
    trajectories = dct_generator.sample(source_samples.reshape(-1, n_pcs), 
                                        source_latent, target_latent,
                                        num_steps=2, return_trajectory=True)

print('sampling done')

In [ ]:
# Plot
plt.figure(figsize=(4, 4))

# Plot true target distribution in background with kdeplot


# Plot trajectories
# for traj in range(trajectories.shape[2]):
#     plt.plot(trajectories[:, 0, traj, 0].cpu(), trajectories[:, 0, traj, 1].cpu(),
#              color='lightblue', alpha=0.5, lw=0.5)
    
#     # Scatter endpoints
#     plt.scatter(trajectories[0, 0, traj, 0].cpu(), trajectories[0, 0, traj, 1].cpu(), 
#                 color='green', s=5, alpha=0.6)
#     plt.scatter(trajectories[-1, 0, traj, 0].cpu(), trajectories[-1, 0, traj, 1].cpu(), 
#                 color='blue', s=5, alpha=0.6)

sns.kdeplot(x=true_target_pcs[:, 0], y=true_target_pcs[:, 1], 
            levels=5, color='lightblue', alpha=0.3, gridsize=20, fill=True)

plt.title(f'{source_donor} to {target_donor}')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

Source donor: GTEX-1HSMQ
Target donor: GTEX-1MCC2
